# Tái hiện ConEm trên Colab free

Bài báo: **ConEm**, Knowledge-Based Systems 329 (2025) 114312.
Repo tác giả: `github.com/anonymous7594/conem` @ `80f0e01` — 133 file, **không có
entry point**, README ghi *"Code will be released soon"*.

---

## Vì sao chạy trên Colab chứ không phải máy cá nhân

Không phải vì nhanh hơn. Vì **nguyên bản hơn**: trên GPU CUDA, hai bản vá
`required` biến mất —

| vá | nội dung | trên T4 |
|---|---|---|
| `b1_mps_device` | thêm nhánh thiết bị MPS | **không cần** |
| `b2_map_location` | tác giả hardcode `map_location='cuda:0'` | **không cần** |

Tác giả viết `device = torch.device("cuda:0" ...)` và `.cuda()` vô điều kiện
(`utils/tools.py:380-407`). Trên T4 những dòng đó đúng **nguyên văn**. Ta đi từ
11 bản vá xuống 9, và hai bản bỏ được đều thuộc nhóm phải biện minh nhiều nhất.

## Thứ tự chạy

Chạy lần lượt từ trên xuống. Nếu session bị cắt: **chạy lại từ cell 1**, mọi thứ
tiếp đúng chỗ dở (checkpoint trên Drive + vá C5 resume + bỏ qua run đã xong).

## 1. Kiểm tra GPU — DỪNG nếu không có

In [ ]:
# Colab free có lúc cấp CPU thay vì GPU. Chạy FEDformer trên CPU là vô ích,
# nên dừng ngay thay vì đốt session.
import torch, subprocess

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device    :', torch.cuda.get_device_name(0))
    print('VRAM      : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    raise SystemExit(
        'KHONG CO GPU. Runtime > Change runtime type > T4 GPU, roi chay lai cell nay.\n'
        'Neu Colab khong cap GPU (het quota), doi roi thu lai session khac.')

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/conem'
for sub in ('checkpoints', 'logs', 'results', 'data', 'code'):
    os.makedirs(os.path.join(DRIVE, sub), exist_ok=True)
print('Drive san sang:', DRIVE)
print(sorted(os.listdir(DRIVE)))

## 3. Đưa code lên Colab

Hai nguồn, tách bạch đúng như kiến trúc ba tầng — **cả hai đều clone, không upload**:

- `ours/` — clone từ repo của chúng ta, `github.com/kkyaha/Forecast`.
- `upstream/conem` — clone từ repo tác giả, pin đúng commit `80f0e01`. Không bao giờ sửa.

`build/conem` sinh ra tại chỗ = upstream + patches. Không nằm trên Drive.

Lưu ý: trong repo của chúng ta, `repro/upstream/conem` là **gitlink không có
`.gitmodules`** nên clone ra rỗng — đó là lý do cell này clone upstream riêng.

In [ ]:
import os, shutil, subprocess

REPRO = '/content/repro'
UP = os.path.join(REPRO, 'upstream', 'conem')
COMMIT = '80f0e01c40e2608b9a838771bb1f308b2aa76ae1'

os.makedirs(REPRO, exist_ok=True)

# --- upstream: clone + pin commit ---
if not os.path.isdir(os.path.join(UP, '.git')):
    os.makedirs(os.path.dirname(UP), exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/anonymous7594/conem', UP],
                   check=True)
subprocess.run(['git', '-C', UP, 'checkout', '-q', COMMIT], check=True)
head = subprocess.run(['git', '-C', UP, 'rev-parse', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', UP, 'status', '--porcelain'],
                       capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f'commit sai: {head}'
assert dirty == '', f'code tac gia bi sua:\n{dirty}'
print('upstream OK :', head[:8], '| status trong = chua bi sua')

# --- ours: clone repo cua chung ta ---
OURS_REPO = 'https://github.com/kkyaha/Forecast.git'
SRC = '/content/Forecast'
if not os.path.isdir(os.path.join(SRC, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', OURS_REPO, SRC], check=True)
else:
    subprocess.run(['git', '-C', SRC, 'pull', '--ff-only'], check=True)

src = os.path.join(SRC, 'repro', 'ours')
dst = os.path.join(REPRO, 'ours')
if not os.path.isdir(src):
    raise SystemExit(f'Thieu {src} -- da push repro/ours/ len GitHub chua?')
if os.path.isdir(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)
n = len([f for _, _, fs in os.walk(dst) for f in fs])
rev = subprocess.run(['git', '-C', SRC, 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip()
print(f'ours OK     : {n} file  @ {rev}')

## 4. Dữ liệu

Cần hai file **của tác giả** (từ `weather.zip` trên Drive của họ) và hai bản
hợp nhất **của chúng ta** (`prepare_data.py` sinh ra):

| file | nguồn |
|---|---|
| `train_weather.csv` | tác giả, 105.039 dòng |
| `test_weather_df_final.csv` | tác giả, 26.281 dòng |
| `weather_all.csv` | của ta, 131.319 dòng = ghép train+test |
| `weather_all_clean.csv` | của ta, đã xử lý `-9999` |

Copy vào `/content` thay vì đọc trực tiếp Drive: Drive I/O chậm, và loader đọc
lại dữ liệu mỗi epoch.

In [ ]:
import os, shutil, subprocess

ddst = os.path.join(REPRO, 'data', 'weather')
dsrc = os.path.join(DRIVE, 'data', 'weather')
os.makedirs(ddst, exist_ok=True)

AUTHOR = ['train_weather.csv', 'test_weather_df_final.csv']
OURS = ['weather_all.csv', 'weather_all_clean.csv']

if not os.path.isdir(dsrc):
    raise SystemExit(
        f'Thieu {dsrc}\n'
        'Upload thu muc repro/data/weather/ cua ban vao Drive tai duong dan tren.')

for f in AUTHOR + OURS:
    s = os.path.join(dsrc, f)
    if os.path.exists(s):
        shutil.copy2(s, os.path.join(ddst, f))
        print('  copy', f, '%.1f MB' % (os.path.getsize(s) / 1e6))

missing = [f for f in AUTHOR if not os.path.exists(os.path.join(ddst, f))]
if missing:
    raise SystemExit(f'Thieu du lieu TAC GIA: {missing}')

# weather_all*.csv dung lai duoc neu da co; khong thi sinh tu train+test
need = [f for f in OURS if not os.path.exists(os.path.join(ddst, f))]
if need:
    print('Thieu', need, '-> chay prepare_data.py')
    subprocess.run(['python', 'ours/analysis/prepare_data.py'], cwd=REPRO, check=True)
    for f in OURS:
        shutil.copy2(os.path.join(ddst, f), os.path.join(dsrc, f))
    print('da sinh va luu nguoc ve Drive')

print()
for f in sorted(os.listdir(ddst)):
    p = os.path.join(ddst, f)
    if os.path.isfile(p):
        print('  %-32s %8.1f MB' % (f, os.path.getsize(p) / 1e6))

## 5. Trỏ checkpoints / logs / results sang Drive

`exp.test()` của tác giả hardcode `'./results/'` (dòng 420, 543 của
`exp_main_compare.py`), nên không truyền tham số được. Dùng symlink.

Nếu không làm bước này, session chết là mất hết.

In [ ]:
import os

for sub in ('checkpoints', 'logs', 'results'):
    local = os.path.join(REPRO, sub)
    target = os.path.join(DRIVE, sub)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.isdir(local):
        # thu muc that (khong phai link) -> doi ten de khong mat du lieu
        os.rename(local, local + '.local')
        print(f'  {sub}/ la thu muc that, da doi thanh {sub}.local')
    os.symlink(target, local)
    print(f'  {sub:12} -> {target}')

## 6. Cài phụ thuộc còn thiếu

In [ ]:
# KHONG pip install torch: Colab da co ban khop CUDA cua no. Cai lai se pha
# hoac mat 10 phut. Chi cai thu Colab thieu.
!pip install -q einops

import einops, torch, numpy, pandas, sklearn
print('einops', einops.__version__, '| torch', torch.__version__,
      '| numpy', numpy.__version__, '| pandas', pandas.__version__)

## 7. Dựng `build/conem` = upstream + patches

In ra từng bản vá, phân loại `required` (không có thì crash) và `choice`
(quyết định của chúng ta, code vẫn chạy nếu bỏ). `build/audit.diff` là diff
thống nhất so với code tác giả.

In [ ]:
import subprocess
r = subprocess.run(['python', 'ours/build.py', '--audit'], cwd=REPRO,
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise SystemExit('build.py that bai')

## 8. Run hiệu chuẩn — đo s/iter thật trên T4

**Không bỏ cell này.** Mọi ước lượng thời gian trước đó suy ra từ Mac
(~2,7 s/iter, ~215 iter/epoch ở stride 24). Chưa ai đo trên T4.

Chạy 1 epoch rồi đọc dòng `speed: ... s/iter`. Nhân với 215 để ra giây/epoch,
rồi quyết định `--budget_min` cho cell kế tiếp.

Dùng `--tag _cal` nên nó **không** đè lên run thật.

In [ ]:
import subprocess
cmd = ['python', 'ours/runner/run.py',
       '--variant', 'conem', '--backbone', 'fedformer',
       '--target', 'T (degC)', '--ext', 'paper',
       '--data_path', 'weather_all_clean.csv',
       '--stride_train', '24', '--train_epochs', '1',
       '--clip_grad', '1.0', '--resume', '1', '--tag', '_cal']
print(' '.join(cmd), '\n')
p = subprocess.Popen(cmd, cwd=REPRO, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
print('\nrc =', p.wait())

## 9. Ma trận thực nghiệm

6 run = 3 cấu hình × 2 target.

| | variant | ext | ý nghĩa |
|---|---|---|---|
| **A** | `conem` | `paper` | tái hiện số liệu chủ đạo |
| **B** | `conem` | `noninfo` | **khử rò rỉ** — cùng kiến trúc, ngoại sinh `wv/wd/rain` ($R^2 < 0{,}1$) |
| **C** | `backbone` | — | baseline bài báo đối chiếu |

Hai target, chọn theo `leakage_test.py`:

| target | $R^2$ từ 6 biến ngoại sinh | vai trò |
|---|---|---|
| `T (degC)` | 1,0000 | đối chiếu được với Bảng 11 |
| `rh (%)` | 0,8809 | kiểm tra cơ chế thật |

**Lập luận trung tâm:** nếu A ≪ C mà B ≈ C thì mức cải thiện đến từ việc
*được biết trước biến quyết định target*, không từ kiến trúc ConEm.

Đặt `--budget_min` theo số đo ở cell 8, chừa biên an toàn. Hết ngân sách nó
dừng **sạch giữa hai run**. Session bị cắt thì chạy lại cell này.

### Dung lượng Drive — đọc trước khi chạy

Mô hình có **116.092.098 tham số**, 86,7% là `complex64`. Đo thực tế:

| file | kích thước |
|---|---|
| `checkpoint.pth` (trọng số tốt nhất) | **0,89 GB** / run |
| `resume.pt` (trọng số + 2 moment Adam) | **2,58 GB** / run |

6 run × 3,5 GB = **21 GB**, vượt 15 GB của Drive free. Nên `run_matrix.py`
**xoá `resume.pt` ngay khi run xong** — chỉ một bản tồn tại tại một thời điểm.
Tổng còn ~6 × 0,89 + 2,6 ≈ **8 GB**. Vẫn phải dọn Drive trước nếu đang chật.

In [ ]:
import subprocess
cmd = ['python', 'ours/runner/run_matrix.py',
       '--checkpoints', DRIVE + '/checkpoints',
       '--logs', DRIVE + '/logs',
       '--results', DRIVE + '/results',
       '--data_path', 'weather_all_clean.csv',
       '--stride_train', '24', '--train_epochs', '10',
       '--clip_grad', '1.0',
       '--tag', '_colab',
       '--budget_min', '150']        # <- sua theo ket qua cell 8
print(' '.join(cmd), '\n')
p = subprocess.Popen(cmd, cwd=REPRO, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
print('\nrc =', p.wait())

## 10. Tổng hợp

`collect_results.py` đọc `results/*/metrics.npy` và dựng bảng đối chiếu.

In [ ]:
import subprocess
r = subprocess.run(['python', 'ours/analysis/collect_results.py'], cwd=REPRO,
                   capture_output=True, text=True)
print(r.stdout[-6000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])

---

## Giới hạn phải khai trong báo cáo

1. **Cấu hình thực nghiệm là của chúng ta, không phải của tác giả.** Repo công bố
   có 0 entry point và 0 giá trị mặc định cho 65 tham số `args.*`. Mọi siêu tham
   số ở đây do chúng ta chọn, đối chiếu Mục 4.2 của bài báo nơi bài có nói.

2. **Số tuyệt đối không so trực tiếp với bài báo.** `stride_train=24` cho ngân
   sách gradient step nhỏ hơn tác giả ~24× mỗi epoch. Giá trị khoa học nằm ở
   so sánh A/B/C **dưới cùng ngân sách**.

3. **`clip_grad=1.0` là thứ chúng ta THÊM** (vá C4). Repo tác giả không có
   gradient clipping ở bất kỳ đâu. Cấu hình đã công bố nổ gradient sau ~250 step
   (0,97 → 3,8 → 27,6 → 391 → 1.909 → 17.777 → 571.064). Đặt `--clip_grad 0`
   để tái hiện đúng hành vi phân kỳ đó.

4. **Ba baseline thuần của tác giả không chạy được** từ code đã công bố:
   `model_fedformer.py`, `model_autoformer.py`, `model_informer.py` đều có
   `os.chdir("/home/ad/20813716/transformer_sales_forecast/model")` ở dòng 4.
   Nhánh C ở đây dùng `exp_main_without_ext` + `model_fedformer` đã vá (A4).
   Nếu cần baseline làm tham chiếu độc lập, nên dùng bản chuẩn của thuml
   `Time-Series-Library` thay vì bản copy của tác giả.

5. **Nửa còn lại của project chưa từng được công bố.** Commit đầu của repo tác giả
   chứa submodule `transformer_sales_forecast` → `6cf0e0d2…`, bị xoá ở commit 2
   (*"delete old folder"*), không có `.gitmodules`. GitHub commit search cho SHA đó:
   0 kết quả. Entry point, requirements và cấu hình thực nghiệm nằm ở đó.